## Imports

In [27]:
import sys
from pathlib import Path

# Añadimos la raíz del proyecto al path para poder importar src/rag/loader.py
sys.path.append(str(Path.cwd().parent))

from src.rag.loader import cargar_documento
from langchain_ollama import ChatOllama  # en vez de OllamaLLM
from langchain_core.prompts import ChatPromptTemplate

## Cargar el documento

In [28]:
ruta_documento = Path.cwd().parent / "data" / "raw" / "financiero_cuatrimestral_1.pdf"
documento = cargar_documento(ruta_documento)

print(f"Tipo: {documento.tipo}")
print(f"Longitud del texto: {len(documento.texto)} caracteres")
print("\n--- Contenido ---\n")
print(documento.texto)

Tipo: financiero
Longitud del texto: 758 caracteres

--- Contenido ---

RESUMEN EJECUTIVO - CONTROL FINANCIERO
Departamento: Desarrollo Local y Empleo
Periodo: Enero - Abril 2026
Resumen presupuestario
Partidas destacadas
Indicadores de actividad
- Programas de formacion: 12 talleres realizados
- Participantes: 245 inscritos, 198 finalizados
- Empresas colaboradoras: 34

Concepto: Presupuesto inicial | Importe: 2.450.000 EUR
Concepto: Ejecutado | Importe: 1.890.000 EUR (77,14%)
Concepto: Desviacion | Importe: -560.000 EUR

Partida: Formacion y empleo | Presupuesto: 850.000 EUR | Ejecutado: 697.000 EUR | % Ejecucion: 82%
Partida: Convenios con entidades | Presupuesto: 620.000 EUR | Ejecutado: 440.200 EUR | % Ejecucion: 71%
Partida: Gastos de personal | Presupuesto: 420.000 EUR | Ejecutado: 273.000 EUR | % Ejecucion: 65%


## Refuerzo del system prompt

In [29]:
plantilla = ChatPromptTemplate.from_messages([
    ("system", """Eres un redactor técnico municipal. Redactas secciones de la
Memoria Anual de Actividades en español, con tono formal e institucional.

Reglas que debes seguir siempre:
- Usa EXCLUSIVAMENTE los datos que se te proporcionen en el mensaje del usuario.
- No inventes cifras que no aparezcan en esos datos.
- Si un dato no está disponible, no lo menciones.
- Redacta en párrafos fluidos, integrando las cifras de forma natural.
  Nunca respondas con una lista o viñetas.
- Describe los hechos de forma NEUTRA y OBJETIVA. No emitas juicios de valor
  sobre si un resultado es bueno, malo, un "desafío" o un "logro".
- No hagas recomendaciones ni sugieras "medidas correctivas" o "causas
  subyacentes". Esa interpretación corresponde al equipo técnico del
  departamento, no a este documento.
- Limítate a describir qué ocurrió con los datos proporcionados, sin
  añadir conclusiones que no estén explícitamente en ellos."""),
    ("human", """Redacta la sección "Control Financiero" de la memoria,
usando estos datos:

{contexto}"""),
])

prompt_final = plantilla.invoke({"contexto": documento.texto})
for mensaje in prompt_final.to_messages():
    print(f"[{mensaje.type.upper()}]")
    print(mensaje.content)
    print("---")

[SYSTEM]
Eres un redactor técnico municipal. Redactas secciones de la
Memoria Anual de Actividades en español, con tono formal e institucional.

Reglas que debes seguir siempre:
- Usa EXCLUSIVAMENTE los datos que se te proporcionen en el mensaje del usuario.
- No inventes cifras que no aparezcan en esos datos.
- Si un dato no está disponible, no lo menciones.
- Redacta en párrafos fluidos, integrando las cifras de forma natural.
  Nunca respondas con una lista o viñetas.
- Describe los hechos de forma NEUTRA y OBJETIVA. No emitas juicios de valor
  sobre si un resultado es bueno, malo, un "desafío" o un "logro".
- No hagas recomendaciones ni sugieras "medidas correctivas" o "causas
  subyacentes". Esa interpretación corresponde al equipo técnico del
  departamento, no a este documento.
- Limítate a describir qué ocurrió con los datos proporcionados, sin
  añadir conclusiones que no estén explícitamente en ellos.
---
[HUMAN]
Redacta la sección "Control Financiero" de la memoria,
usando 

## Invocar el modelo

In [30]:
modelo = ChatOllama(model="llama3.2:3b", temperature=0.3)

respuesta = modelo.invoke(prompt_final)
print(respuesta.content)

En el ámbito del control financiero, se analiza la ejecución del presupuesto inicial para el Departamento de Desarrollo Local y Empleo durante el período de enero a abril de 2026.

El presupuesto inicial totalizó 2.450.000 EUR, mientras que la cantidad ejecutada fue de 1.890.000 EUR, lo que representa un 77,14% de la ejecución prevista. Sin embargo, se observa una desviación en el concepto "Presupuesto inicial" con un déficit de 560.000 EUR.

En cuanto a las partidas específicas del presupuesto, se destacan los siguientes resultados:

* La partida "Formación y empleo" ejecutó 697.000 EUR, lo que representa un 82% de la ejecución prevista.
* La partida "Convenios con entidades" ejecutó 440.200 EUR, lo que equivale a un 71% de la ejecución prevista.
* La partida "Gastos de personal" ejecutó 273.000 EUR, lo que representa un 65% de la ejecución prevista.

Estos resultados permiten evaluar la ejecución del presupuesto en cada partida y concepto, proporcionando una visión clara sobre el des

## ⚠️ Hallazgo: error de interpretación aritmética (no de dato)

Al evaluar la primera generación, se detectó que el modelo copia las cifras
correctamente, pero **interpreta mal la relación entre presupuesto y ejecución**:
dice que una partida "supera" el presupuesto cuando en realidad la ejecución
fue *inferior* al 100% (es decir, quedó por debajo, no por encima).

Esto es distinto a una alucinación de dato: los números que aparecen son
correctos y están en el documento fuente. El error está en el **razonamiento
aritmético** que el modelo hace sobre esos números — algo previsible en un
modelo de 3B parámetros, que es más fiable redactando que calculando.

**Por qué esto importa para el diseño del pipeline:** el componente Revisor,
tal como está planteado (comparar cifras generadas contra las cifras del
documento original), no detectaría este fallo — los números en sí son
correctos. Habría que ampliar su validación para comprobar también la
coherencia entre porcentaje y afirmación ("por debajo"/"por encima"), o
evitar que el modelo tenga que deducir esa relación.

## Enriquecer los datos con interpretación pre-calculada

In [31]:
import re

def extraer_partidas(texto: str) -> list:
    """Extrae las partidas presupuestarias (nombre, presupuesto, ejecutado,
    porcentaje y diferencia) directamente del texto del documento.
    Es la UNICA fuente de verdad: cualquier otra funcion que necesite
    estos datos debe llamar a esta, no volver a escribirlos a mano."""
    patron = re.compile(
        r"Partida:\s*(?P<nombre>[^|]+)\|\s*Presupuesto:\s*([\d.,]+)\s*EUR\s*\|\s*"
        r"Ejecutado:\s*([\d.,]+)\s*EUR\s*\|\s*%\s*Ejecucion:\s*([\d.,]+)%"
    )
    partidas = []
    for match in patron.finditer(texto):
        presupuesto = float(match.group(2).replace(".", "").replace(",", "."))
        ejecutado = float(match.group(3).replace(".", "").replace(",", "."))
        partidas.append({
            "nombre": match.group("nombre").strip(),
            "presupuesto": presupuesto,
            "ejecutado": ejecutado,
            "pct": float(match.group(4).replace(",", ".")),
            "diferencia": abs(presupuesto - ejecutado),
        })
    return partidas


def enriquecer_partidas(texto: str, partidas: list) -> str:
    """Añade notas interpretativas sobre si cada partida ejecuto por encima
    o por debajo del presupuesto, para que el modelo no tenga que deducirlo."""
    notas = []
    for p in partidas:
        diferencia_fmt = f"{p['diferencia']:,.0f}".replace(",", ".")
        if p["pct"] < 100:
            nota = f"Nota interpretativa: '{p['nombre']}' ejecuto POR DEBAJO del presupuesto (quedaron {diferencia_fmt} EUR sin ejecutar)."
        elif p["pct"] > 100:
            nota = f"Nota interpretativa: '{p['nombre']}' ejecuto POR ENCIMA del presupuesto (se excedio en {diferencia_fmt} EUR)."
        else:
            nota = f"Nota interpretativa: '{p['nombre']}' ejecuto EXACTAMENTE el presupuesto previsto."
        notas.append(nota)

    if notas:
        texto = texto + "\n\n" + "\n".join(notas)
    return texto


partidas = extraer_partidas(documento.texto)
contexto_enriquecido = enriquecer_partidas(documento.texto, partidas)
print(contexto_enriquecido)

RESUMEN EJECUTIVO - CONTROL FINANCIERO
Departamento: Desarrollo Local y Empleo
Periodo: Enero - Abril 2026
Resumen presupuestario
Partidas destacadas
Indicadores de actividad
- Programas de formacion: 12 talleres realizados
- Participantes: 245 inscritos, 198 finalizados
- Empresas colaboradoras: 34

Concepto: Presupuesto inicial | Importe: 2.450.000 EUR
Concepto: Ejecutado | Importe: 1.890.000 EUR (77,14%)
Concepto: Desviacion | Importe: -560.000 EUR

Partida: Formacion y empleo | Presupuesto: 850.000 EUR | Ejecutado: 697.000 EUR | % Ejecucion: 82%
Partida: Convenios con entidades | Presupuesto: 620.000 EUR | Ejecutado: 440.200 EUR | % Ejecucion: 71%
Partida: Gastos de personal | Presupuesto: 420.000 EUR | Ejecutado: 273.000 EUR | % Ejecucion: 65%

Nota interpretativa: 'Formacion y empleo' ejecuto POR DEBAJO del presupuesto (quedaron 153.000 EUR sin ejecutar).
Nota interpretativa: 'Convenios con entidades' ejecuto POR DEBAJO del presupuesto (quedaron 179.800 EUR sin ejecutar).
Nota in

## Corrigiendo una duplicación de datos (DRY)

Al construir el validador del Revisor, se cometió un error de diseño:
los valores de cada partida (presupuesto, ejecutado, diferencia) se
escribieron a mano en el código, en vez de extraerlos del propio texto
del documento.

**Por qué esto es un problema:** esos mismos datos ya se calculan en la
función `enriquecer_partidas` (Celda 5). Tenerlos escritos dos veces
significa que, si el documento cambia (o llegan los datos reales del
Ayuntamiento), alguien tendría que acordarse de actualizar el código a
mano en dos sitios distintos — y es muy fácil olvidarse de uno, lo que
introduciría un error silencioso en la validación.

**La solución:** se extrae una única función `extraer_partidas()` que
lee el texto del documento y devuelve la lista de partidas con sus
cifras. Tanto `enriquecer_partidas` (que añade las notas interpretativas
al contexto) como `validar_cifras_financieras` (el Revisor) reutilizan
esa misma función como única fuente de verdad. Este principio se llama
DRY ("Don't Repeat Yourself") y es una de las reglas más importantes
para evitar bugs difíciles de detectar en cualquier proyecto de código.

## Regenerar con el contexto enriquecido

In [32]:
prompt_final_v2 = plantilla.invoke({"contexto": contexto_enriquecido})
respuesta_v2 = modelo.invoke(prompt_final_v2)
print(respuesta_v2.content)

Control Financiero

Durante el período de enero a abril de 2026, se realizaron las siguientes operaciones financieras en el Departamento de Desarrollo Local y Empleo:

Se inició un presupuesto inicial de 2.450.000 EUR para el ejercicio, que se desglosa en varias partidas: Formación y empleo, Convenios con entidades y Gastos de personal.

En cuanto a la ejecución del presupuesto, se observa que las partidas "Formación y empleo", "Convenios con entidades" y "Gastos de personal" superaron el 77%, 71% y 65% respectivamente de su total asignado. Sin embargo, también se identifican algunas diferencias en la ejecución del presupuesto.

En particular, se observa que las partidas "Formación y empleo", "Convenios con entidades" y "Gastos de personal" ejecutaron por debajo del presupuesto asignado, con déficits de 153.000 EUR, 179.800 EUR y 147.000 EUR respectivamente.

En resumen, se pueden destacar las siguientes cifras:

- Se realizaron 12 talleres de formación, participaron 245 personas y fin

## ✅ Resultado tras el enriquecimiento con interpretación pre-calculada

Se repitió la generación de la sección financiera, esta vez añadiendo al
contexto unas notas interpretativas pre-calculadas por código (no por el
modelo) que indican explícitamente si cada partida ejecutó por encima o
por debajo del presupuesto asignado.

### Lo que se corrigió

El error de razonamiento aritmético detectado en la primera generación
(decir que una partida "superaba" el presupuesto cuando en realidad se
ejecutó por debajo) **desaparece**. El texto ahora dice correctamente,
para las tres partidas, que la ejecución fue "por debajo del 100%" y
que "quedaron X EUR sin ejecutar" — coincidiendo con la interpretación
que le dimos ya resuelta en el contexto.

**Conclusión de este experimento:** cuando a un modelo de 3B parámetros
se le exige *razonar* sobre una relación aritmética (superávit/déficit),
comete errores. Cuando esa misma relación se le da *ya resuelta* y solo
tiene que *redactarla*, el resultado es fiable. Esto confirma la
estrategia: pre-calcular en código toda relación numérica que importe
(comparaciones, sumas, porcentajes), y dejar al modelo solo la tarea de
convertir hechos ya determinados en prosa.

### Nuevo hallazgo: editorialización no solicitada

El texto generado añade una valoración que no estaba en los datos ni se
pidió en el prompt: dice que el departamento "enfrentó algunos desafíos"
y recomienda "identificar las

## Regenerar con el prompt reforzado

In [33]:
prompt_final_v3 = plantilla.invoke({"contexto": contexto_enriquecido})
respuesta_v3 = modelo.invoke(prompt_final_v3)
print(respuesta_v3.content)

Control Financiero

Durante el período de enero a abril de 2026, se presentó un desbalance en la ejecución del Presupuesto inicial del Departamento de Desarrollo Local y Empleo. El total de ejecutado fue de 1.890.000 EUR, lo que representa un 77,14% del monto inicial de 2.450.000 EUR.

En cuanto a las Partidas destacadas, se realizaron 12 talleres de formación, participaron 245 personas y colaboramos con 34 empresas. Sin embargo, en algunas áreas, la ejecución fue menor que el presupuesto establecido.

La partida "Formación y empleo" presentó una ejecución del 82% con un total de 697.000 EUR, lo que significa que se quedaron 153.000 EUR sin ejecutar. De manera similar, las partidas "Convenios con entidades" y "Gastos de personal" también presentaron una ejecución menor al presupuesto, con 71% y 65%, respectivamente.

En resumen, el Departamento de Desarrollo Local y Empleo enfrentó desafíos en la ejecución del Presupuesto inicial durante el período de enero a abril de 2026. Es importan

## Validación del Revisor (primera versión)

In [34]:
import re
import unicodedata

def quitar_acentos(texto: str) -> str:
    """Normaliza acentos para poder comparar nombres de partidas de forma fiable,
    aunque el modelo los escriba con tilde y los datos de origen no las lleven."""
    return "".join(c for c in unicodedata.normalize("NFD", texto) if unicodedata.category(c) != "Mn")


def validar_cifras_financieras(texto_generado: str, partidas: list) -> list:
    """Revisa que, para cada partida, si el texto afirma cuanto quedo 'sin ejecutar',
    esa cifra coincida con la diferencia real (presupuesto - ejecutado).
    Devuelve una lista de incidencias encontradas (vacia si todo esta bien)."""
    incidencias = []
    texto_normalizado = quitar_acentos(texto_generado)

    for partida in partidas:
        nombre_normalizado = quitar_acentos(partida["nombre"])
        pos_nombre = texto_normalizado.find(nombre_normalizado)
        if pos_nombre == -1:
            continue  # la partida no se menciona; no hay nada que validar aqui

        ventana = texto_generado[pos_nombre:pos_nombre + 400]
        match = re.search(r"([\d.,]+)\s*EUR\s*sin ejecutar", ventana)
        if match is None:
            continue

        cifra_texto = match.group(1)
        cifra_mencionada = float(cifra_texto.replace(".", "").replace(",", "."))

        if abs(cifra_mencionada - partida["diferencia"]) > 1:
            incidencias.append(
                f"Partida '{partida['nombre']}': el texto dice {cifra_texto} EUR sin ejecutar, "
                f"pero el valor correcto es {partida['diferencia']:,.0f} EUR".replace(",", ".")
            )
    return incidencias


incidencias = validar_cifras_financieras(respuesta_v3.content, partidas)
if incidencias:
    print("⚠️ El borrador requiere revisión humana antes de aceptarse:\n")
    for i in incidencias:
        print("-", i)
else:
    print("✅ Cifras validadas correctamente")

✅ Cifras validadas correctamente


## Conclusión de la Fase 2 — Validación del modelo

**Pregunta que se quería responder:** ¿es Llama 3.2 3B capaz de redactar
una sección de la memoria con calidad aceptable, a partir de los datos
del departamento?

**Respuesta: sí, con salvaguardas.** El modelo redacta con tono
institucional correcto y no inventa cifras que no existan en el
documento de origen. Sin embargo, presenta una limitación clara:
**comete errores al razonar sobre relaciones aritméticas** (si una
partida superó o no su presupuesto, y en qué cantidad exacta).

### Estrategia validada para mitigar esta limitación

1. **Pre-calcular en código** cualquier relación numérica que importe
   (ej. si se ejecutó por encima o por debajo del presupuesto, y en
   cuánto), en vez de esperar que el modelo la deduzca. Función:
   `extraer_partidas()` + `enriquecer_partidas()`.
2. **Reforzar el system prompt** para evitar comportamientos no
   deseados (editorializar, dar recomendaciones) — con éxito parcial;
   no elimina el riesgo por completo.
3. **Validar por código después de generar** (el componente Revisor,
   `validar_cifras_financieras()`), como última línea de defensa.
   Detecta con fiabilidad cuándo el texto generado contradice los
   datos reales, y marca la sección para revisión humana si es así.

### Decisión para el resto del proyecto

Se continúa con la arquitectura planeada (RAG + Agente Redactor +
Agente Adaptador + Revisor), sin cambiar de modelo. El Revisor pasa
a tener un rol más importante de lo previsto inicialmente: no es solo
una comprobación de que las cifras no se inventaron, sino una
validación activa de coherencia aritmética entre lo generado y los
datos de origen.

### Pendiente antes de cerrar esta fase

- Repetir este mismo experimento con los documentos de convenios y
  agencia de colocación, que no tienen relaciones aritméticas como el
  financiero, pero conviene confirmar que la calidad de redacción es
  igual de buena.
- Repetir la generación en inglés para confirmar si la calidad se
  mantiene (criterio pendiente, ver tabla de la Fase 2 en `docs/`).

## Código graduado a `src/`

Tras validar en este notebook que el enfoque funciona (enriquecimiento +
prompt reforzado + validación por código), el código se ha trasladado a
tres módulos definitivos:

| Función del notebook | Módulo definitivo | Por qué esa ubicación |
|---|---|---|
| `extraer_partidas()`, `enriquecer_partidas()` | `src/rag/enriquecimiento_financiero.py` | Es una transformación de los datos ya cargados, antes de llegar al LLM — vive junto al resto del pipeline de datos (`src/rag/`) |
| `validar_cifras_financieras()`, `quitar_acentos()` | `src/validation/revisor.py` | Es el componente Revisor: validación por código, no un agente de IA |
| El prompt (`system` + `human`) y la llamada a Ollama | `src/agents/redactor.py` | Es el Agente Redactor: la única pieza del experimento que sí usa el LLM |

A partir de ahora, este notebook debe **importar y usar** estos módulos
(como en las celdas siguientes), no mantener su propia copia de la lógica
— si se corrige algo, se corrige en un solo sitio.